In [ ]:
import numpy as np
from ortools.linear_solver import pywraplp


def solve_convex_combo_ortools(A: np.ndarray, t: np.ndarray):
    """
    A: 形状 (3, k) の numpy 配列
    t: 形状 (3,) の numpy 配列
    戻り値: w (numpy 配列, shape=(k,)) 非負かつ sum(w)=1 の最適解
             成功すれば (w, solver_status) を返し、
             失敗すれば (None, status) を返す
    """
    m, k = A.shape  # m = 3, 点は A の行数、k は塗料の数

    # (1) G, g0 を作成
    #    G = 2 * (A^T A), g0 = -2 * (A^T t)
    ATA = A.T.dot(A)        # 形状 (k, k)
    ATt = A.T.dot(t)        # 形状 (k,)

    G = 2.0 * ATA           # QP の二次項係数行列
    g0 = -2.0 * ATt         # QP の一次項係数ベクトル

    # (2) MPSolver を作成  (バックエンドに SCIP を指定)
    #     SCIP が使える環境なら "SCIP" を指定して凸QP を解く
    solver = pywraplp.Solver.CreateSolver('SCIP')
    if solver is None:
        print("SCIP ソルバが利用できません。インストールを確認してください。")
        return None, None

    # (3) 変数 w_i を作成（連続変数、下限 0、上限 1 としておく）
    w = [solver.NumVar(0.0, 1.0, f"w[{i}]") for i in range(k)]
    # （上限 1 にしているのは「必ず sum(w)=1 を満たす」が前提なので十分杞憂不要ですが、
    # もし w_i <= 1 を明示したければ 1 にしておくと過変数の数値が暴れにくくなります）

    # (4) 制約 Sum_i w_i = 1
    ct = solver.RowConstraint(1.0, 1.0, "sum_eq_one")
    for i in range(k):
        ct.SetCoefficient(w[i], 1.0)

    # (5) 目的関数（二次形式）を設定
    #     OR-Tools では MPObjective().SetQuadraticCoefficient(var_i, var_j, coeff)
    #     で二次項を与えられる。線形項は SetCoefficient(w[i], g0[i]) で与える。
    objective = solver.Objective()
    # (5-1) 線形項 g0^T w
    for i in range(k):
        objective.SetCoefficient(w[i], g0[i])

    # (5-2) 二次項 (1/2) w^T (G) w  → OR-Tools は「(1/2) を自前で取る」ために
    #       q_ij を直接渡すと「objective には 1/2 x^T Q x + …」と解釈される。
    #       つまり OR-Tools 側は `obj = 0.5 * sum_{i,j} Q[i,j] * w[i]*w[j] + linear terms` になる。
    #       だから、ここには Q = G_full（すでに 2*A^T A なので、OR-Tools に渡すと 0.5*(2 A^T A) = (A^T A) の形になる）。
    #       よって「G をそのまま SetQuadraticCoefficient」に渡してよい。
    for i in range(k):
        for j in range(k):
            if G[i, j] != 0.0:
                objective.SetQuadraticCoefficient(w[i], w[j], G[i, j])
    objective.SetMinimization()

    # (6) ソルバを実行
    status = solver.Solve()
    if status != pywraplp.Solver.OPTIMAL:
        print(f"Solver failed with status {status}")
        return None, status

    # (7) 解を numpy 配列に取り出して返却
    w_opt = np.array([w[i].solution_value() for i in range(k)])
    return w_opt, status


if __name__ == "__main__":
    # --- (A) 例：手持ち 4 本の塗料ベクトルを用意 ---
    #     それぞれ CMY の 3 次元ベクトル
    paint_list = np.array([
        [1.0, 0.0, 0.0],   # #0
        [0.0, 1.0, 0.0],   # #1
        [0.0, 0.0, 1.0],   # #2
        [0.5, 0.5, 0.0],   # #3
    ]).T  # shape (3,4)

    # --- (B) ある部分集合を選んで A (3×k) を作る例 ----
    #     ここでは {0,1,2} の組み合わせを固定
    indices = [0, 1, 2]
    A = paint_list[:, indices]   # shape (3,3)

    # --- (C) ターゲット色 t (3,) を用意 ---
    #     3 本を等量 (1/3,1/3,1/3) で混ぜた色
    t = (paint_list[:, 0] + paint_list[:, 1] + paint_list[:, 2]) / 3.0

    # --- (D) QP を解く ---
    w_opt, status = solve_convex_combo_ortools(A, t)
    if w_opt is not None:
        print("Optimal w:", w_opt)
        print("Sum(w) =", w_opt.sum())
        # 再構成誤差も確認
        c_hat = A.dot(w_opt)
        print("Reconstructed color:", c_hat)
        print("Error:", np.linalg.norm(c_hat - t) ** 2)
    else:
        print("No solution found.")
